In [3]:
import re
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import pandas as pd
from transformers import AutoModel, AutoTokenizer
from underthesea import word_tokenize
from pyvi import ViTokenizer

e:\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "../../"))

In [2]:
def clean_text(text):
    text = re.sub(r"\d+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"[^\w\s]", "", text)
    return text

def hybrid_tokenize(text):
    ut = word_tokenize(text, format="text").split()
    pv = ViTokenizer.tokenize(text).split()
    merged, i, j = [], 0, 0
    while i < len(ut) and j < len(pv):
        if ut[i] == pv[j]:
            merged.append(ut[i]); i+=1; j+=1
        else:
            if len(ut[i]) >= len(pv[j]):
                merged.append(ut[i]); i+=1; j+=len(pv[j].split('_'))
            else:
                merged.append(pv[j]); j+=1; i+=len(ut[i].split('_'))
    merged += ut[i:] + pv[j:]
    return merged

In [3]:
class PhoBERTClassifier(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.phobert = AutoModel.from_pretrained("vinai/phobert-base")
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.phobert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        out = self.phobert(input_ids=input_ids, attention_mask=attention_mask)
        cls_vec = out.last_hidden_state[:, 0, :]
        return self.classifier(self.dropout(cls_vec))

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_path = os.path.join(BASE_DIR, "src/nlp/phobert_sentiment_model.pth")
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base", use_fast=False)
model = PhoBERTClassifier().to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

PhoBERTClassifier(
  (phobert): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(64001, 768, padding_idx=1)
      (position_embeddings): Embedding(258, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): L

In [11]:
LABELS = ["Vui vẻ", "Tức giận", "Buồn bã", "Sợ hãi", "Trung lập"]

def predict(text, max_len=256):
    cleaned = clean_text(text)
    tokens  = hybrid_tokenize(cleaned)
    enc     = tokenizer(
        " ".join(tokens),
        truncation=True,
        padding="max_length",
        max_length=max_len,
        return_tensors="pt"
    )
    input_ids = enc["input_ids"].to(device)
    mask      = enc["attention_mask"].to(device)
    with torch.no_grad():
        logits = model(input_ids, attention_mask=mask)
        probs  = torch.softmax(logits, dim=1)
        idx    = probs.argmax(dim=1).item()
        conf   = probs[0, idx].item()
    return LABELS[idx], conf

In [16]:
samples = [
    "Hôm nay mình rất vui và hào hứng!",
    "Mình đang lo lắng về kỳ thi cuối kỳ.",
    "Tôi tức giận với bạn",
    "Tôi thấy buồn và cô đơn."
]

for s in samples:
    label, conf = predict(s)
    print(f"\"{s}\"  →  {label} (confidence: {conf:.2f})")

"Hôm nay mình rất vui và hào hứng!"  →  Vui vẻ (confidence: 0.87)
"Mình đang lo lắng về kỳ thi cuối kỳ."  →  Sợ hãi (confidence: 0.95)
"Tôi tức giận với bạn"  →  Tức giận (confidence: 0.81)
"Tôi thấy buồn và cô đơn."  →  Buồn bã (confidence: 0.92)
